In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [41]:
df = pd.read_csv("C:\\Users\\wasim\\Downloads\\movie_recommendation_system(1).csv")

In [42]:
df.head()

,movie_id,title,genres,keywords,release_year,rating
0,1,Inception,Sci-Fi|Thriller|Action,Dream|Mind|Technology|Heist,2010,8.8
1,2,Interstellar,Sci-Fi|Drama|Adventure,Space|Time|Science|Family,2014,8.7
2,3,The Matrix,Sci-Fi|Action|Thriller,Technology|Virtual Reality|AI|Rebellion,1999,8.7
3,4,The Dark Knight,Action|Crime|Drama,Batman|Joker|Justice|Crime,2008,9.0
4,5,Avengers: Endgame,Action|Sci-Fi|Adventure,Superhero|Time Travel|Team|Battle,2019,8.4


In [43]:
df.shape

(25, 6)

In [44]:
df.columns

Index(['movie_id', 'title', 'genres', 'keywords', 'release_year', 'rating'], dtype='str')

In [45]:
df.info

<bound method DataFrame.info of     movie_id                                  title  \
0          1                              Inception   
1          2                           Interstellar   
2          3                             The Matrix   
3          4                        The Dark Knight   
4          5                      Avengers: Endgame   
5          6                               Iron Man   
6          7                Spider-Man: No Way Home   
7          8                         Doctor Strange   
8          9                Guardians of the Galaxy   
9         10                          Jurassic Park   
10        11                                 Avatar   
11        12                          Black Panther   
12        13  Harry Potter and the Sorcerer's Stone   
13        14                  The Lord of the Rings   
14        15                            The Martian   
15        16                                Gravity   
16        17                     

In [46]:
df.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
release_year    0
rating          0
dtype: int64

In [47]:
df.isnull().describe()

,movie_id,title,genres,keywords,release_year,rating
count,25,25,25,25,25,25
unique,1,1,1,1,1,1
top,False,False,False,False,False,False
freq,25,25,25,25,25,25


In [48]:
df.describe

<bound method NDFrame.describe of     movie_id                                  title  \
0          1                              Inception   
1          2                           Interstellar   
2          3                             The Matrix   
3          4                        The Dark Knight   
4          5                      Avengers: Endgame   
5          6                               Iron Man   
6          7                Spider-Man: No Way Home   
7          8                         Doctor Strange   
8          9                Guardians of the Galaxy   
9         10                          Jurassic Park   
10        11                                 Avatar   
11        12                          Black Panther   
12        13  Harry Potter and the Sorcerer's Stone   
13        14                  The Lord of the Rings   
14        15                            The Martian   
15        16                                Gravity   
16        17                   

In [49]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn Names:")
print(df.columns)
print("\nDataset Information:")
df.info()

Rows: 25
Columns: 6

Column Names:
Index(['movie_id', 'title', 'genres', 'keywords', 'release_year', 'rating'], dtype='str')

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      25 non-null     int64  
 1   title         25 non-null     str    
 2   genres        25 non-null     str    
 3   keywords      25 non-null     str    
 4   release_year  25 non-null     int64  
 5   rating        25 non-null     float64
dtypes: float64(1), int64(2), str(3)
memory usage: 2.9 KB


In [50]:
df = df.fillna("")

In [51]:
df.isnull().sum()

movie_id        0
title           0
genres          0
keywords        0
release_year    0
rating          0
dtype: int64

In [52]:
df["combined_features"] = (
    df["genres"].astype(str) + " " +
    df["keywords"].astype(str)
)
df[["title", "combined_features"]].head()

,title,combined_features
0,Inception,Sci-Fi|Thriller|Action Dream|Mind|Technology|H...
1,Interstellar,Sci-Fi|Drama|Adventure Space|Time|Science|Family
2,The Matrix,Sci-Fi|Action|Thriller Technology|Virtual Real...
3,The Dark Knight,Action|Crime|Drama Batman|Joker|Justice|Crime
4,Avengers: Endgame,Action|Sci-Fi|Adventure Superhero|Time Travel|...


In [53]:
df["combined_features"] = df["combined_features"].str.replace("|", " ", regex=False)
df[["title", "combined_features"]].head()

,title,combined_features
0,Inception,Sci-Fi Thriller Action Dream Mind Technology H...
1,Interstellar,Sci-Fi Drama Adventure Space Time Science Family
2,The Matrix,Sci-Fi Action Thriller Technology Virtual Real...
3,The Dark Knight,Action Crime Drama Batman Joker Justice Crime
4,Avengers: Endgame,Action Sci-Fi Adventure Superhero Time Travel ...


In [54]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["combined_features"])
print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

TF-IDF Matrix Shape:
(25, 66)


In [55]:
cosine_sim = cosine_similarity(tfidf_matrix)
print("Cosine Similarity Shape:")
print(cosine_sim.shape)

Cosine Similarity Shape:
(25, 25)


In [56]:
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

In [63]:
def recommend_movies(movie_title, num_recommendations=5):
    
    if movie_title not in indices:
        return "Movie not found in dataset."
    
    movie_index = indices[movie_title]
    
    similarity_scores = list(enumerate(cosine_sim[movie_index]))
    
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    similarity_scores = similarity_scores[1:num_recommendations + 1]
    
    movie_indices = [i[0] for i in similarity_scores]
    
    result = df.iloc[movie_indices][
        ["title", "genres", "release_year", "rating"]
    ].copy()
    
    result["similarity_score"] = [
        round(i[1], 3) for i in similarity_scores
    ]
    
    return result

In [64]:
recommend_movies("Inception")

,title,genres,release_year,rating,similarity_score
2,The Matrix,Sci-Fi|Action|Thriller,1999,8.7,0.323
15,Gravity,Sci-Fi|Drama|Thriller,2013,7.7,0.308
11,Black Panther,Action|Adventure|Sci-Fi,2018,7.3,0.253
5,Iron Man,Action|Sci-Fi|Adventure,2008,7.9,0.248
10,Avatar,Action|Adventure|Sci-Fi,2009,7.8,0.229


In [65]:
recommend_movies("Intersteller")

'Movie not found in dataset.'

In [66]:
recommend_movies("Iron Man")

,title,genres,release_year,rating,similarity_score
11,Black Panther,Action|Adventure|Sci-Fi,2018,7.3,0.473
2,The Matrix,Sci-Fi|Action|Thriller,1999,8.7,0.434
8,Guardians of the Galaxy,Action|Sci-Fi|Comedy,2014,8.0,0.362
4,Avengers: Endgame,Action|Sci-Fi|Adventure,2019,8.4,0.318
10,Avatar,Action|Adventure|Sci-Fi,2009,7.8,0.311


In [ ]:
recommend_movies("Spider-Man No Way Home")

In [41]:
recommend_movies("Inception", 3)

,title,genres,release_year,rating,similarity_score
2,The Matrix,Sci-Fi|Action|Thriller,1999,8.7,0.323
15,Gravity,Sci-Fi|Drama|Thriller,2013,7.7,0.308
11,Black Panther,Action|Adventure|Sci-Fi,2018,7.3,0.253


In [42]:
recommend_movies("Inception", 10)

,title,genres,release_year,rating,similarity_score
2,The Matrix,Sci-Fi|Action|Thriller,1999,8.7,0.323
15,Gravity,Sci-Fi|Drama|Thriller,2013,7.7,0.308
11,Black Panther,Action|Adventure|Sci-Fi,2018,7.3,0.253
5,Iron Man,Action|Sci-Fi|Adventure,2008,7.9,0.248
10,Avatar,Action|Adventure|Sci-Fi,2009,7.8,0.229
9,Jurassic Park,Adventure|Sci-Fi|Thriller,1993,8.2,0.209
24,Ready Player One,Sci-Fi|Action|Adventure,2018,7.4,0.209
8,Guardians of the Galaxy,Action|Sci-Fi|Comedy,2014,8.0,0.161
7,Doctor Strange,Action|Sci-Fi|Fantasy,2016,7.5,0.158
4,Avengers: Endgame,Action|Sci-Fi|Adventure,2019,8.4,0.142


In [44]:
similarity_df = pd.DataFrame(
    cosine_sim,
    index=df["title"],
    columns=df["title"]
)
similarity_df.head()

title,Inception,Interstellar,The Matrix,The Dark Knight,Avengers: Endgame,Iron Man,Spider-Man: No Way Home,Doctor Strange,Guardians of the Galaxy,Jurassic Park,...,Gravity,Toy Story,Finding Nemo,The Lion King,Frozen,Titanic,The Notebook,La La Land,The Social Network,Ready Player One
title,,,,,,,,,,,,,,,,,,,,,
Inception,1.000000,0.103053,0.322565,0.042475,0.141539,0.247760,0.127569,0.157687,0.161404,0.209047,...,0.307885,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.082848,0.208622
Interstellar,0.103053,1.000000,0.099209,0.057482,0.303253,0.170747,0.273322,0.288328,0.345815,0.328000,...,0.360058,0.219303,0.324297,0.264838,0.184802,0.073734,0.200811,0.057290,0.083002,0.143775
The Matrix,0.322565,0.099209,1.000000,0.040891,0.136259,0.433733,0.122811,0.151805,0.155384,0.201250,...,0.296401,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.079758,0.529595
The Dark Knight,0.042475,0.057482,0.040891,1.000000,0.043363,0.049473,0.039083,0.048310,0.049449,0.000000,...,0.051360,0.000000,0.000000,0.053270,0.000000,0.046446,0.049703,0.036088,0.052284,0.041658
Avengers: Endgame,0.141539,0.303253,0.136259,0.043363,1.000000,0.317834,0.529554,0.411313,0.524592,0.123413,...,0.094001,0.078594,0.074105,0.041194,0.042229,0.000000,0.000000,0.000000,0.000000,0.174270


In [45]:
similarity_df["Inception"].sort_values(ascending=False).head(10)

title
Inception                  1.000000
The Matrix                 0.322565
Gravity                    0.307885
Black Panther              0.253008
Iron Man                   0.247760
Avatar                     0.228643
Jurassic Park              0.209047
Ready Player One           0.208622
Guardians of the Galaxy    0.161404
Doctor Strange             0.157687
Name: Inception, dtype: float64

In [46]:
recommendation = recommend_movies("Inception", 5)
recommendation

,title,genres,release_year,rating,similarity_score
2,The Matrix,Sci-Fi|Action|Thriller,1999,8.7,0.323
15,Gravity,Sci-Fi|Drama|Thriller,2013,7.7,0.308
11,Black Panther,Action|Adventure|Sci-Fi,2018,7.3,0.253
5,Iron Man,Action|Sci-Fi|Adventure,2008,7.9,0.248
10,Avatar,Action|Adventure|Sci-Fi,2009,7.8,0.229


In [50]:
def movie_recommender(movie_title, n=5):
    if movie_title not in indices:
        print("Movie not found.")
        print("\nAvailable movies:")
        print(df["title"].to_string(index=False))
        return
    movie_index = indices[movie_title]
    scores = list(enumerate(cosine_sim[movie_index]))
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )
    top_scores = scores[1:n+1]
    print(f"\nRecommendations for: {movie_title}\n")
    for rank, (index, score) in enumerate(top_scores, start=1):
        movie = df.iloc[index]
         print(
            f"{rank}. {movie['title']} "
            f"| Rating: {movie['rating']} "
            f"| Similarity: {score:.3f}"
        )

In [51]:
movie_recommender("Inception")


Recommendations for: Inception

1. The Matrix | Rating: 8.7 | Similarity: 0.323
2. Gravity | Rating: 7.7 | Similarity: 0.308
3. Black Panther | Rating: 7.3 | Similarity: 0.253
4. Iron Man | Rating: 7.9 | Similarity: 0.248
5. Avatar | Rating: 7.8 | Similarity: 0.229


In [52]:
movie_recommender("Avengers")

Movie not found.

Available movies:
                            Inception
                         Interstellar
                           The Matrix
                      The Dark Knight
                    Avengers: Endgame
                             Iron Man
              Spider-Man: No Way Home
                       Doctor Strange
              Guardians of the Galaxy
                        Jurassic Park
                               Avatar
                        Black Panther
Harry Potter and the Sorcerer's Stone
                The Lord of the Rings
                          The Martian
                              Gravity
                            Toy Story
                         Finding Nemo
                        The Lion King
                               Frozen
                              Titanic
                         The Notebook
                           La La Land
                   The Social Network
                     Ready Player One


In [53]:
movie_recommender("Bat Man")

Movie not found.

Available movies:
                            Inception
                         Interstellar
                           The Matrix
                      The Dark Knight
                    Avengers: Endgame
                             Iron Man
              Spider-Man: No Way Home
                       Doctor Strange
              Guardians of the Galaxy
                        Jurassic Park
                               Avatar
                        Black Panther
Harry Potter and the Sorcerer's Stone
                The Lord of the Rings
                          The Martian
                              Gravity
                            Toy Story
                         Finding Nemo
                        The Lion King
                               Frozen
                              Titanic
                         The Notebook
                           La La Land
                   The Social Network
                     Ready Player One


In [54]:
movie_recommender("Toy Story")


Recommendations for: Toy Story

1. Finding Nemo | Rating: 8.2 | Similarity: 0.606
2. Frozen | Rating: 7.4 | Similarity: 0.482
3. The Lion King | Rating: 8.5 | Similarity: 0.470
4. Guardians of the Galaxy | Rating: 8.0 | Similarity: 0.310
5. Interstellar | Rating: 8.7 | Similarity: 0.219


In [55]:
movie_recommender("Titanic")


Recommendations for: Titanic

1. The Notebook | Rating: 7.8 | Similarity: 0.368
2. La La Land | Rating: 8.0 | Similarity: 0.267
3. Interstellar | Rating: 8.7 | Similarity: 0.074
4. The Lion King | Rating: 8.5 | Similarity: 0.068
5. The Social Network | Rating: 7.8 | Similarity: 0.067
